# Assignment: Introduction to Transformers
## Two Real-World Use Cases: Sentiment Analysis & Text Generation | HuggingFace + PyTorch

---

### Assignment Objectives

By completing this assignment you will:

1. **Explain** the core ideas behind transformers: self-attention, positional encoding, and encoder/decoder architecture.
2. **Use** pretrained transformer models from HuggingFace for real NLP tasks — no from-scratch implementation needed.
3. **Fine-tune** a DistilBERT model on the IMDB movie review dataset for **sentiment analysis** (Use Case 1).
4. **Generate** text with a pretrained GPT-2 model and explore how temperature and top-k sampling control output (Use Case 2).
5. **Compare** encoder-based (BERT) vs decoder-based (GPT) transformer architectures and explain when to use each.
6. **Visualize** attention weights to understand what the model "looks at" when making predictions.

> **Deliverable:** After running every cell, you will have all the figures, tables, and analysis needed to write a professional lab report.

---

### Report Structure Guide

Your report should follow this structure (each section maps to a part of this notebook):

| Report Section | What to Write | Notebook Section |
|---|---|---|
| **1. Introduction** | What are transformers? Attention, encoders, decoders — why they replaced RNNs | Part 1 |
| **2. Setup & Tools** | HuggingFace ecosystem: models, tokenizers, pipelines | Part 2 |
| **3. Use Case 1** | Sentiment analysis: IMDB dataset, DistilBERT fine-tuning, evaluation | Parts 3–6 |
| **4. Use Case 2** | Text generation: GPT-2, sampling strategies, temperature effects | Parts 7–9 |
| **5. Comparison** | Encoder vs decoder architectures, when to use which | Part 10 |
| **6. Conclusion** | Key takeaways, limitations, real-world applications | Part 11 |

### Grading Rubric

| Category | Points | Description |
|---|---|---|
| Transformer Concepts | 15 | Clear explanation of attention, positional encoding, encoder/decoder |
| Sentiment Analysis | 30 | Data loading, tokenization, fine-tuning, evaluation with metrics |
| Text Generation | 25 | GPT-2 generation, sampling strategies, temperature analysis |
| Visualization & Analysis | 20 | Attention heatmaps, confusion matrix, comparison table |
| Report | 10 | Clear writing, figures, analysis, conclusions |
| **Total** | **100** | |

---

*Apeiron AI | "Boundless Possibilities, Infinite Potential"*
*© 2026 | www.aperionaiml.com*

---

# Part 1: Background — What Are Transformers?

## 1.1 The Problem with RNNs

Before transformers, we used **RNNs** (Recurrent Neural Networks) for sequential data like text. RNNs have two major problems:

```
RNN: Process words one at a time (slow!)

   "I"  →  "love"  →  "deep"  →  "learning"
    ↓        ↓         ↓          ↓
   h₁   →   h₂   →   h₃   →    h₄

Problem 1: Sequential — can't parallelize (slow on GPUs)
Problem 2: Long sequences → early words get "forgotten"
```

## 1.2 The Transformer Solution: Attention

Transformers (Vaswani et al., 2017 — *"Attention Is All You Need"*) solve both problems with **self-attention**:

```
Transformer: Process ALL words at once (parallel!)

   "I"    "love"    "deep"    "learning"
    ↓        ↓         ↓          ↓
  ┌─────────────────────────────────┐
  │     Self-Attention Layer        │
  │  Every word attends to every    │
  │  other word simultaneously      │
  └─────────────────────────────────┘
    ↓        ↓         ↓          ↓
   out₁     out₂      out₃       out₄
```

**Self-attention** lets each word "look at" every other word to understand context:
- *"The bank of the **river**"* → bank = riverbank
- *"The **money** in the bank"* → bank = financial institution

## 1.3 Two Types of Transformers

```
┌───────────────────────────────────────────────────────────┐
│           TRANSFORMER FAMILY                        │
├─────────────────────────────┬─────────────────────────────┤
│    ENCODER (BERT-style)      │    DECODER (GPT-style)      │
│                              │                              │
│  Reads ENTIRE text at once   │  Generates text one token    │
│  Understands context         │  at a time (left-to-right)   │
│                              │                              │
│  Best for:                   │  Best for:                   │
│  - Classification            │  - Text generation           │
│  - Sentiment analysis        │  - Story writing             │
│  - Named entity recognition  │  - Code completion           │
│  - Question answering        │  - Chatbots                  │
│                              │                              │
│  Examples: BERT, DistilBERT, │  Examples: GPT-2, GPT-4,     │
│  RoBERTa, ALBERT             │  LLaMA, Claude               │
└─────────────────────────────┴─────────────────────────────┘
```

In this assignment we will use **both**:
- **Use Case 1 (Sentiment Analysis):** DistilBERT (encoder) — reads a movie review and classifies it as positive/negative
- **Use Case 2 (Text Generation):** GPT-2 (decoder) — takes a prompt and generates a continuation

---

# Part 2: Environment Setup

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  INSTALL DEPENDENCIES (run once)                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
%pip install torch transformers datasets matplotlib seaborn scikit-learn numpy pandas --quiet

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  IMPORTS                                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import time
import textwrap

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    pipeline,
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ── Plotting defaults ────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})
sns.set_style("whitegrid")

# ── Device selection ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version      : {torch.__version__}")
print(f"Device               : {device}")
if device.type == "cuda":
    print(f"GPU                  : {torch.cuda.get_device_name(0)}")

import transformers
print(f"Transformers version : {transformers.__version__}")

## 2.1 The HuggingFace Ecosystem

HuggingFace provides three key tools:

| Tool | What It Does | Example |
|---|---|---|
| **Tokenizer** | Converts text → numbers (tokens) | `"I love it"` → `[101, 1045, 2293, 2009, 102]` |
| **Model** | Pretrained neural network | DistilBERT, GPT-2 |
| **Pipeline** | One-line inference (tokenizer + model) | `pipeline("sentiment-analysis")("I love it")` |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  QUICK DEMO: ONE-LINE SENTIMENT ANALYSIS                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# This is how easy HuggingFace makes it!
quick_classifier = pipeline("sentiment-analysis", device=device)

demo_texts = [
    "I absolutely loved this movie! The acting was incredible.",
    "This was the worst film I have ever seen. Complete waste of time.",
    "The movie was okay, nothing special but not terrible either.",
]

print("Quick Demo: One-Line Sentiment Analysis")
print("=" * 60)
for text in demo_texts:
    result = quick_classifier(text)[0]
    print(f"\n  Text: \"{text[:60]}...\"")
    print(f"  Label: {result['label']}  |  Confidence: {result['score']:.4f}")

del quick_classifier  # free memory

> **For your report:** Explain what a pipeline is. Why is it powerful that we can do sentiment analysis in one line of code? What pretrained knowledge does the model already have?

---

# USE CASE 1: Sentiment Analysis with DistilBERT

---

# Part 3: The IMDB Dataset

## 3.1 About the Dataset

The **IMDB** dataset contains 50,000 movie reviews labeled as **positive** or **negative**. It is the standard benchmark for sentiment analysis.

| Split | Samples | Task |
|---|---|---|
| Train | 25,000 | Fine-tune the model |
| Test | 25,000 | Evaluate performance |

We will use a **subset** (2,000 train + 1,000 test) to keep training fast.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LOAD IMDB DATASET (subset)                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Load full IMDB dataset
imdb = load_dataset("imdb")

# Use a subset to keep training fast (2000 train, 1000 test)
TRAIN_SIZE = 2000
TEST_SIZE = 1000

train_data = imdb["train"].shuffle(seed=42).select(range(TRAIN_SIZE))
test_data  = imdb["test"].shuffle(seed=42).select(range(TEST_SIZE))

print(f"Train samples: {len(train_data)}")
print(f"Test samples:  {len(test_data)}")
print(f"Labels: 0 = Negative, 1 = Positive")
print(f"\nLabel distribution (train):")
labels = train_data["label"]
print(f"  Positive: {sum(labels)} ({sum(labels)/len(labels)*100:.1f}%)")
print(f"  Negative: {len(labels) - sum(labels)} ({(len(labels)-sum(labels))/len(labels)*100:.1f}%)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  EXPLORE SAMPLE REVIEWS                                                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print("Sample Reviews")
print("=" * 70)
for i in range(4):
    review = train_data[i]
    sentiment = "POSITIVE" if review["label"] == 1 else "NEGATIVE"
    text_preview = review["text"][:200].replace("\n", " ")
    print(f"\n[{sentiment}] Review {i+1}:")
    print(f"  \"{text_preview}...\"")
    print(f"  Length: {len(review['text'])} characters")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  VISUALIZE REVIEW LENGTHS                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

lengths = [len(r["text"].split()) for r in train_data]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Word count distribution
ax1.hist(lengths, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax1.axvline(x=np.median(lengths), color='red', linestyle='--', linewidth=2,
            label=f'Median: {int(np.median(lengths))} words')
ax1.set_xlabel('Number of Words')
ax1.set_ylabel('Count')
ax1.set_title('Review Length Distribution')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Label balance
label_counts = [len(labels) - sum(labels), sum(labels)]
ax2.bar(['Negative (0)', 'Positive (1)'], label_counts,
        color=['#e74c3c', '#2ecc71'], edgecolor='black')
ax2.set_ylabel('Count')
ax2.set_title('Label Distribution')
for i, v in enumerate(label_counts):
    ax2.text(i, v + 10, str(v), ha='center', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('asgn_fig_imdb_data_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: asgn_fig_imdb_data_overview.png')

> **For your report:** Describe the IMDB dataset. How many samples? What is the label distribution? Why is balanced data important for classification? Include the data overview figure.

---

# Part 4: Tokenization — Text to Numbers

Transformers cannot read text directly. A **tokenizer** converts text into numbers (token IDs) that the model understands.

```
"I love this movie"  →  Tokenizer  →  [101, 1045, 2293, 2023, 3185, 102]
                                          [CLS]  I    love  this  movie [SEP]
```

- **[CLS]** = special "classification" token (model uses this for predictions)
- **[SEP]** = separator / end-of-sequence token

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  TOKENIZER DEMO                                                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Demo: see how tokenization works
sample_text = "I absolutely loved this movie! The acting was incredible."
tokens = tokenizer(sample_text)

print("Tokenization Demo")
print("=" * 60)
print(f"Original text : {sample_text}")
print(f"Token IDs     : {tokens['input_ids']}")
print(f"Attention mask: {tokens['attention_mask']}")
print(f"Number of tokens: {len(tokens['input_ids'])}")

# Decode back to see individual tokens
decoded_tokens = tokenizer.convert_ids_to_tokens(tokens['input_ids'])
print(f"\nTokens: {decoded_tokens}")
print(f"\nVocabulary size: {tokenizer.vocab_size:,} tokens")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  TOKENIZE THE FULL DATASET                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

MAX_LENGTH = 256  # truncate long reviews to 256 tokens

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

# Tokenize train and test sets
train_tokenized = train_data.map(tokenize_function, batched=True)
test_tokenized  = test_data.map(tokenize_function, batched=True)

# Set format for PyTorch
train_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print(f"Tokenized train: {len(train_tokenized)} samples")
print(f"Tokenized test:  {len(test_tokenized)} samples")
print(f"Max length:      {MAX_LENGTH} tokens")
print(f"\nSample tokenized entry:")
print(f"  input_ids shape:     {train_tokenized[0]['input_ids'].shape}")
print(f"  attention_mask shape: {train_tokenized[0]['attention_mask'].shape}")
print(f"  label:               {train_tokenized[0]['label']}")

> **For your report:** Explain what tokenization does. Why do we need `padding` and `truncation`? What is the `attention_mask` for? Why 256 tokens instead of using the full review?

---

# Part 5: Fine-Tuning DistilBERT for Sentiment Analysis

## 5.1 What is Fine-Tuning?

**Fine-tuning** = taking a pretrained model and training it a little more on your specific task.

```
Pretrained DistilBERT           Your Fine-Tuned Model
(trained on Wikipedia +         (specialized for IMDB
 BookCorpus = general            sentiment analysis)
 English understanding)
          │                              │
          │   + IMDB training data       │
          │   + a few epochs             │
          │   + classification head      │
          └───────────────────────────┘
```

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LOAD PRETRAINED DistilBERT                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Load DistilBERT with a classification head (2 classes: positive/negative)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)
sentiment_model = sentiment_model.to(device)

# Count parameters
total_params = sum(p.numel() for p in sentiment_model.parameters())
trainable_params = sum(p.numel() for p in sentiment_model.parameters() if p.requires_grad)

print(f"Model: {MODEL_NAME}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1e6:.0f} MB (float32)")
print(f"\nModel architecture:")
print(sentiment_model)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  FINE-TUNE DistilBERT                                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Create DataLoaders
BATCH_SIZE = 16
train_loader = DataLoader(train_tokenized, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_tokenized, batch_size=BATCH_SIZE, shuffle=False)

# Optimizer (smaller LR for fine-tuning — we don't want to destroy pretrained knowledge)
optimizer = optim.AdamW(sentiment_model.parameters(), lr=2e-5, weight_decay=0.01)

NUM_EPOCHS = 3
train_losses = []
train_accs = []
test_accs = []

print(f"Fine-tuning {MODEL_NAME} for {NUM_EPOCHS} epochs...")
print(f"Batch size: {BATCH_SIZE} | LR: 2e-5 | Device: {device}")
print("-" * 60)

for epoch in range(NUM_EPOCHS):
    # --- Training ---
    sentiment_model.train()
    epoch_loss = 0.0
    correct = 0
    total = 0
    start_time = time.time()

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        # Forward pass — HuggingFace models return loss automatically
        outputs = sentiment_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss
        logits = outputs.logits

        # Backward + update
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = epoch_loss / len(train_loader)
    train_acc = correct / total
    train_losses.append(avg_loss)
    train_accs.append(train_acc)

    # --- Evaluation ---
    sentiment_model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            outputs = sentiment_model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=-1)
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)

    test_acc = test_correct / test_total
    test_accs.append(test_acc)
    elapsed = time.time() - start_time

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Loss: {avg_loss:.4f} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Test Acc: {test_acc:.4f} | "
          f"Time: {elapsed:.1f}s")

print("-" * 60)
print(f"Final Test Accuracy: {test_accs[-1]:.4f} ({test_accs[-1]*100:.1f}%)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  PLOT TRAINING CURVES                                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

epochs_range = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(epochs_range, train_losses, 'o-', color='steelblue', linewidth=2,
         markersize=8, label='Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs_range, train_accs, 'o-', color='steelblue', linewidth=2,
         markersize=8, label='Train Accuracy')
ax2.plot(epochs_range, test_accs, 's-', color='darkorange', linewidth=2,
         markersize=8, label='Test Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training & Test Accuracy')
ax2.set_ylim(0.5, 1.0)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('asgn_fig_sentiment_training.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: asgn_fig_sentiment_training.png')

> **For your report:** Include the training curves. How quickly does the model converge? Why do we use a very small learning rate (2e-5) for fine-tuning? What would happen with a large learning rate like 0.01?

---

# Part 6: Evaluation — Sentiment Analysis Results

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  FULL EVALUATION ON TEST SET                                                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Collect all predictions
all_preds = []
all_labels = []

sentiment_model.eval()
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"]
        outputs = sentiment_model(input_ids=input_ids, attention_mask=attention_mask)
        preds = outputs.logits.argmax(dim=-1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

# Classification report
print("Classification Report")
print("=" * 60)
print(classification_report(
    all_labels, all_preds,
    target_names=["Negative", "Positive"],
    digits=4,
))

overall_acc = accuracy_score(all_labels, all_preds)
print(f"Overall Accuracy: {overall_acc:.4f} ({overall_acc*100:.1f}%)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CONFUSION MATRIX                                                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'], ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title(f'Confusion Matrix (Accuracy: {overall_acc:.1%})', fontsize=13)

plt.tight_layout()
plt.savefig('asgn_fig_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: asgn_fig_confusion_matrix.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  TEST ON YOUR OWN REVIEWS                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

custom_reviews = [
    "This movie was absolutely fantastic! Best film of the year.",
    "Terrible acting, awful plot, do not waste your time.",
    "It was alright. Some good moments but overall forgettable.",
    "A masterpiece of storytelling with breathtaking visuals.",
    "I fell asleep halfway through. So boring.",
    "Not great, not terrible. Just a mediocre film.",
]

print("Testing on Custom Reviews")
print("=" * 70)

sentiment_model.eval()
for review in custom_reviews:
    inputs = tokenizer(review, return_tensors="pt", truncation=True,
                       max_length=MAX_LENGTH, padding="max_length").to(device)
    with torch.no_grad():
        outputs = sentiment_model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    pred = probs.argmax().item()
    label = "POSITIVE" if pred == 1 else "NEGATIVE"
    confidence = probs[0][pred].item()

    print(f"\n  \"{review}\"")
    print(f"  Prediction: {label} (confidence: {confidence:.4f})")

> **For your report:** Report accuracy, precision, recall, and F1-score for both classes. Include the confusion matrix. How does the model handle ambiguous reviews? What types of reviews does it misclassify?

---

# USE CASE 2: Text Generation with GPT-2

---

# Part 7: Introduction to GPT-2

## 7.1 How GPT-2 Works

GPT-2 is a **decoder-only** transformer. It generates text one token at a time, always looking **left** (at previous tokens):

```
Prompt:  "The cat sat on"

Step 1:  "The cat sat on" → GPT-2 → "the"     (most likely next word)
Step 2:  "The cat sat on the" → GPT-2 → "mat"
Step 3:  "The cat sat on the mat" → GPT-2 → "."

Result:  "The cat sat on the mat."
```

The model assigns a **probability** to every word in its vocabulary, then **samples** from those probabilities.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LOAD GPT-2                                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

GPT_MODEL = "gpt2"  # 124M parameters (smallest GPT-2)

gpt_tokenizer = AutoTokenizer.from_pretrained(GPT_MODEL)
gpt_model = AutoModelForCausalLM.from_pretrained(GPT_MODEL)
gpt_model = gpt_model.to(device)
gpt_model.eval()

# GPT-2 has no padding token by default; set it
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

# Count parameters
gpt_params = sum(p.numel() for p in gpt_model.parameters())
print(f"Model: {GPT_MODEL}")
print(f"Parameters: {gpt_params:,}")
print(f"Vocabulary: {gpt_tokenizer.vocab_size:,} tokens")
print(f"Model size: ~{gpt_params * 4 / 1e6:.0f} MB")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  BASIC TEXT GENERATION                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def generate_text(prompt, max_new_tokens=80, temperature=1.0, top_k=50,
                  do_sample=True):
    """Generate text from a prompt using GPT-2."""
    inputs = gpt_tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = gpt_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            do_sample=do_sample,
            pad_token_id=gpt_tokenizer.eos_token_id,
        )
    return gpt_tokenizer.decode(outputs[0], skip_special_tokens=True)


# Try several prompts
prompts = [
    "Artificial intelligence will change the world because",
    "Once upon a time in a small village, there lived",
    "The most important thing about machine learning is",
    "In the year 2050, robots",
]

print("Basic Text Generation with GPT-2")
print("=" * 70)

for prompt in prompts:
    generated = generate_text(prompt)
    print(f"\nPrompt: \"{prompt}\"")
    print(f"Output: {textwrap.fill(generated, width=70)}")
    print("-" * 70)

> **For your report:** Describe how GPT-2 generates text token by token. Are the outputs coherent? Do they stay on topic? What limitations do you notice?

---

# Part 8: Controlling Generation — Temperature & Sampling

## 8.1 What is Temperature?

**Temperature** controls how "creative" vs "predictable" the output is:

```
Temperature = 0.1  →  Very confident, repetitive, safe
Temperature = 1.0  →  Balanced (default)
Temperature = 1.5  →  Creative, surprising, sometimes nonsensical
```

Technically, temperature scales the logits before softmax:
- Low temp → probabilities become more "peaked" (top word gets almost all probability)
- High temp → probabilities become more "flat" (rare words get more chance)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  TEMPERATURE EXPERIMENT                                                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

prompt = "The future of artificial intelligence is"
temperatures = [0.3, 0.7, 1.0, 1.5]

print(f"Prompt: \"{prompt}\"")
print("=" * 70)

temp_outputs = {}
for temp in temperatures:
    output = generate_text(prompt, temperature=temp, max_new_tokens=60)
    temp_outputs[temp] = output
    print(f"\nTemperature = {temp}:")
    print(f"  {textwrap.fill(output, width=65, subsequent_indent='  ')}")
    print("-" * 70)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  VISUALIZE TEMPERATURE EFFECT ON PROBABILITIES                               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Show how temperature changes the probability distribution
sample_prompt = "The cat sat on the"
inputs = gpt_tokenizer(sample_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = gpt_model(**inputs)
    logits = outputs.logits[0, -1, :]  # logits for next token

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
temps = [0.3, 0.7, 1.0, 1.5]

for ax, temp in zip(axes, temps):
    probs = torch.softmax(logits / temp, dim=-1).cpu()
    top_probs, top_ids = probs.topk(10)
    top_words = [gpt_tokenizer.decode(idx) for idx in top_ids]

    colors = ['#e74c3c' if i == 0 else '#3498db' for i in range(10)]
    ax.barh(range(10), top_probs.numpy(), color=colors, edgecolor='black')
    ax.set_yticks(range(10))
    ax.set_yticklabels(top_words, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Probability')
    ax.set_title(f'T = {temp}', fontsize=12)
    ax.set_xlim(0, max(0.5, top_probs[0].item() * 1.1))
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Next-Token Probabilities for \"{sample_prompt} ___\"',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('asgn_fig_temperature_effect.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: asgn_fig_temperature_effect.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  TOP-K SAMPLING EXPERIMENT                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

prompt = "Deep learning is a subset of machine learning that"
top_k_values = [5, 20, 50, 200]

print(f"Prompt: \"{prompt}\"")
print("=" * 70)

for k in top_k_values:
    output = generate_text(prompt, top_k=k, temperature=0.8, max_new_tokens=60)
    print(f"\ntop_k = {k}:")
    print(f"  {textwrap.fill(output, width=65, subsequent_indent='  ')}")
    print("-" * 70)

print("\nGreedy (no sampling, top_k=1):")
output = generate_text(prompt, do_sample=False, max_new_tokens=60)
print(f"  {textwrap.fill(output, width=65, subsequent_indent='  ')}")

> **For your report:** Explain how temperature and top-k sampling affect text generation. Include the temperature probability figure. When would you want low temperature (e.g., translation)? When would you want high temperature (e.g., creative writing)?

---

# Part 9: Visualizing Attention

Attention weights show which words the model "pays attention to" when processing each word. This is what makes transformers interpretable.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  ATTENTION HEATMAP (DistilBERT)                                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Get attention weights from DistilBERT
attn_text = "This movie was absolutely terrible and I hated every minute of it"
inputs_attn = tokenizer(attn_text, return_tensors="pt").to(device)

sentiment_model.eval()
with torch.no_grad():
    outputs_attn = sentiment_model(
        **inputs_attn,
        output_attentions=True,
    )

# Get attention from last layer, first head
# Shape: (num_layers, batch, num_heads, seq_len, seq_len)
attention = outputs_attn.attentions[-1][0]  # last layer, first sample
tokens_display = tokenizer.convert_ids_to_tokens(inputs_attn["input_ids"][0])

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for idx, ax in enumerate(axes.flat):
    if idx >= attention.shape[0]:
        ax.axis('off')
        continue
    attn_weights = attention[idx].cpu().numpy()
    sns.heatmap(attn_weights, xticklabels=tokens_display,
                yticklabels=tokens_display, cmap='Blues',
                ax=ax, cbar=True, square=True)
    ax.set_title(f'Head {idx + 1}', fontsize=11)
    ax.tick_params(axis='both', labelsize=7)

plt.suptitle(f'Attention Weights (Last Layer) for: \"{attn_text}\"',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('asgn_fig_attention_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: asgn_fig_attention_heatmap.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  AVERAGE ATTENTION PER TOKEN                                                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Average attention across all heads → which tokens get the most attention?
avg_attention = attention.mean(dim=0).cpu().numpy()  # average over heads
# Sum attention RECEIVED by each token (columns)
token_importance = avg_attention.sum(axis=0)  # how much each token is attended to
token_importance = token_importance / token_importance.sum()  # normalize

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#e74c3c' if imp > np.mean(token_importance) else '#3498db'
          for imp in token_importance]
ax.bar(range(len(tokens_display)), token_importance, color=colors, edgecolor='black')
ax.set_xticks(range(len(tokens_display)))
ax.set_xticklabels(tokens_display, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Attention Received (normalized)')
ax.set_title(f'Token Importance via Attention — \"{attn_text}\"')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('asgn_fig_token_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: asgn_fig_token_importance.png')

> **For your report:** Include the attention heatmap and token importance figures. Which words does the model pay most attention to? Does this match your intuition about which words carry sentiment? Why do some heads attend to different patterns?

---

# Part 10: Encoder vs Decoder — Comparison & Analysis

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  ARCHITECTURE COMPARISON TABLE                                               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

comparison = pd.DataFrame({
    "Property": [
        "Architecture", "Attention Type", "Direction", "Parameters",
        "Best For", "Example Task", "Used In This Assignment",
    ],
    "DistilBERT (Encoder)": [
        "Encoder-only", "Bidirectional (sees all tokens)",
        "Full context (left + right)",
        f"{sum(p.numel() for p in sentiment_model.parameters()):,}",
        "Understanding / Classification",
        "Sentiment Analysis",
        "Use Case 1: IMDB classification",
    ],
    "GPT-2 (Decoder)": [
        "Decoder-only", "Causal / Masked (sees only left)",
        "Left-to-right only",
        f"{sum(p.numel() for p in gpt_model.parameters()):,}",
        "Generation / Completion",
        "Text Generation",
        "Use Case 2: Text generation",
    ],
})

print("Encoder vs Decoder Comparison")
print("=" * 80)
print(comparison.to_string(index=False))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  PARAMETER COMPARISON CHART                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

bert_params = sum(p.numel() for p in sentiment_model.parameters()) / 1e6
gpt2_params = sum(p.numel() for p in gpt_model.parameters()) / 1e6

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Parameter count comparison
models = ['DistilBERT', 'GPT-2']
params = [bert_params, gpt2_params]
colors = ['#3498db', '#e74c3c']
bars = ax1.bar(models, params, color=colors, edgecolor='black', width=0.5)
ax1.set_ylabel('Parameters (Millions)')
ax1.set_title('Model Size Comparison')
for bar, p in zip(bars, params):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{p:.0f}M', ha='center', fontweight='bold')
ax1.grid(True, alpha=0.3)

# Architecture diagram (simplified)
ax2.axis('off')
ax2.set_title('Architecture Summary', fontsize=13)

arch_text = (
    "ENCODER (DistilBERT):\n"
    "\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n"
    "Input \u2192 [CLS] + tokens + [SEP]\n"
    "      \u2192 Embedding + Position\n"
    "      \u2192 6 Transformer Layers\n"
    "      \u2192 [CLS] representation\n"
    "      \u2192 Classification Head\n"
    "      \u2192 Positive / Negative\n\n"
    "DECODER (GPT-2):\n"
    "\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n"
    "Input \u2192 tokens\n"
    "      \u2192 Embedding + Position\n"
    "      \u2192 12 Transformer Layers\n"
    "        (causal mask: can't\n"
    "         look at future tokens)\n"
    "      \u2192 Next-token prediction\n"
    "      \u2192 Generated text"
)
ax2.text(0.1, 0.95, arch_text, transform=ax2.transAxes,
         fontsize=10, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#f0f0f0', alpha=0.8))

plt.tight_layout()
plt.savefig('asgn_fig_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: asgn_fig_model_comparison.png')

> **For your report:** Compare encoder and decoder architectures. What is bidirectional vs causal attention? Why can't you use GPT-2 for sentiment analysis as easily as BERT? Why can't you use BERT for text generation? Include the comparison chart.

---

# Part 11: Save & Conclusion

## 11.1 Save the Sentiment Model

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SAVE FINE-TUNED MODEL                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

save_dir = "sentiment_model_distilbert"
sentiment_model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Sentiment model saved to: {save_dir}/")
print(f"Test Accuracy: {overall_acc:.4f} ({overall_acc*100:.1f}%)")
print(f"\nTo reload later:")
print(f"  model = AutoModelForSequenceClassification.from_pretrained('{save_dir}')")
print(f"  tokenizer = AutoTokenizer.from_pretrained('{save_dir}')")

## 11.2 Summary of All Saved Figures

Every figure generated by this notebook is saved as a PNG for your report:

| Figure File | Content | Report Section |
|---|---|---|
| `asgn_fig_imdb_data_overview.png` | Review length distribution + label balance | Dataset |
| `asgn_fig_sentiment_training.png` | Fine-tuning loss and accuracy curves | Use Case 1: Training |
| `asgn_fig_confusion_matrix.png` | Confusion matrix with accuracy | Use Case 1: Evaluation |
| `asgn_fig_temperature_effect.png` | Next-token probabilities at 4 temperatures | Use Case 2: Temperature |
| `asgn_fig_attention_heatmap.png` | Multi-head attention weights heatmap | Attention Visualization |
| `asgn_fig_token_importance.png` | Token importance via average attention | Attention Visualization |
| `asgn_fig_model_comparison.png` | DistilBERT vs GPT-2 parameter comparison | Comparison |

## 11.3 Report Writing Guide

### What to Write in Each Section

**1. Introduction (1 page)**
- What are transformers? Why did they replace RNNs?
- Explain self-attention in simple terms (every word looks at every other word)
- Encoder vs decoder: when to use which
- What is HuggingFace and why is it important?

**2. Use Case 1: Sentiment Analysis (1.5 pages)**
- Describe the IMDB dataset; include `asgn_fig_imdb_data_overview.png`
- Explain tokenization: what it does, padding, truncation, special tokens
- What is fine-tuning? Why use a small learning rate?
- Include `asgn_fig_sentiment_training.png` — discuss convergence
- Include `asgn_fig_confusion_matrix.png` — report accuracy, precision, recall, F1
- Test on custom reviews — how well does it generalize?

**3. Use Case 2: Text Generation (1 page)**
- Explain how GPT-2 generates text (autoregressive, left-to-right)
- Temperature: what it is, how it affects output quality
- Include `asgn_fig_temperature_effect.png` — discuss the trade-off
- Top-k sampling: what it is, how it limits the vocabulary
- Show sample generated texts at different settings

**4. Attention Visualization (0.5 page)**
- Include `asgn_fig_attention_heatmap.png` and `asgn_fig_token_importance.png`
- Which tokens get the most attention? Does it match intuition?
- What do different attention heads focus on?

**5. Comparison (0.5 page)**
- Include `asgn_fig_model_comparison.png`
- Encoder vs decoder: attention type, directionality, use cases
- Which architecture for which task?

**6. Conclusion (0.5 page)**
- Key takeaways about transformers
- Real-world applications (ChatGPT, translation, search, etc.)
- Limitations of small models (GPT-2 vs GPT-4)
- What you learned from this assignment

---

<center>

### Assignment Complete

**Remember to:**
1. Run all cells from top to bottom before submission
2. Ensure all outputs and figures are visible
3. Write your report using the generated figures
4. Save your fine-tuned model

---

*Apeiron AI | "Boundless Possibilities, Infinite Potential"*
*© 2026 | www.aperionaiml.com*

</center>